In [1]:
%run init_env.py

Added to Python path: C:\git\cs5305
Environment initialization completed successfully.


In [2]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = create_azure_llm("chat")
large_model.name = "Large Context Model"

standard_model = create_azure_llm("nano")
standard_model.name = "Standard Model"



@wrap_model_call #Marks this as a LangChain middleware that intercepts and modifies model requests/responses
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    # Override the model for this request
    request = request.override(model=model)  

    #this modified request is then passed to the next handler in the chain, which will ultimately call the selected model
    
    return handler(request)

Creating Azure OpenAI LLM
  Deployment: lums-gpt-4.1-mini
  Endpoint: https://aoai-foundry-swc.openai.azure.com/
  API Version: 2025-01-01-preview
  Temperature: 0.7
Creating Azure OpenAI LLM
  Deployment: gpt-5.4-nano
  Endpoint: https://aoai-foundry-swc.openai.azure.com/
  API Version: 2025-01-01-preview
  Temperature: 0.7


In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model=large_model,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

I didn’t—I'm not able to physically water the office plants myself.  

If you want, tell me which plant it is (or where it is), and I can remind you what to check (soil dryness, how much water, and whether it needs draining).


In [5]:
print(response["messages"][-1].response_metadata['model_name'])

gpt-5.4-nano-2026-03-17


In [6]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

We should consider repotting the plant when its roots start to grow out of the drainage holes or if it becomes root-bound, which usually happens every 12 to 18 months. I'll keep an eye on it and let you know when it's time!


In [7]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-4.1-mini-2025-04-14
